# Experiment - Quantifying the guest-profile outcome leakage

> **Takeaway (full-data run, 2026-06-12) -** The leak bought us ~0.10-0.12 AUC:
> HistGB val-AUC drops 0.9172 -> 0.8197 and LogReg 0.8828 -> 0.7655 without the three
> profile fields. The leaky models assign **82-88% cancel probability** to
> profile-missing rows - the state ~38% of upcoming bookings are in at scoring time,
> i.e. the production forecast would explode. BUT: on the filled-only slice the fields
> still add **+0.056-0.062 real AUC** - nationality/language carry genuine signal, so
> booking-time snapshotting is worth building to re-admit clean versions later.
> Interim verdict: the three fields must come OUT of every model roster.

## The finding (cancel_rate_by_feature, 2026-06-12)
`primaryGuest_address_countryCode` (-> `guest_country_region`), `preferredLanguage`
and (milder) `travelPurpose` are completed at/around **check-in**. Cancelled bookings
freeze with empty profiles (country missing: 31% of Canceled vs 1.4% of CheckedOut),
while **38% of future Confirmed bookings** - the scoring population - have empty
country *right now*. So in training data "profile missing" secretly means "cancelled",
and at scoring time it just means "hasn't arrived yet". Classic outcome leakage via
the data-capture process.

## Three numbers this notebook produces
1. **Offline inflation** - AUC/AP with vs without the three fields (train -> val).
   How much of our reported performance was leak-assisted?
2. **Production damage** - what the leaky model predicts for profile-missing rows.
   38% of upcoming bookings would receive this score; their plausible true cancel
   propensity is nowhere near it.
3. **Recoverable signal** - on the val slice where the profile IS filled, does the
   with-profile model beat the without-model? If yes, nationality carries real signal
   worth recovering later via booking-time snapshots.

**Test-vault note:** everything here trains on **train** and evaluates on **val**.
The temporal test block stays untouched - this is a diagnostic, not a final score.


In [1]:
# ---- setup ----
import sys
from pathlib import Path
_here = Path.cwd().resolve()
while not (_here / "pyproject.toml").exists() and _here != _here.parent:
    _here = _here.parent
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

import warnings
import numpy as np
import pandas as pd

from src.data_loader import load_clean_reservations
from src.features import model_feature_roster
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

TARGET = "status"
LEAKY  = ["guest_country_region", "primaryGuest_preferredLanguage", "travelPurpose"]

def _obj(frame):
    if isinstance(frame, pd.Series): frame = frame.to_frame()
    return frame.astype("string").to_numpy(dtype=object, na_value=np.nan)

def make_matrix(df_tr, df_ev, num, cat):
    ni = SimpleImputer(strategy="median").fit(df_tr[num])
    sc = StandardScaler().fit(ni.transform(df_tr[num]))
    Xtr = [sc.transform(ni.transform(df_tr[num]))]; Xev = [sc.transform(ni.transform(df_ev[num]))]
    ci = SimpleImputer(strategy="most_frequent").fit(_obj(df_tr[cat]))
    oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(ci.transform(_obj(df_tr[cat])))
    Xtr.append(oh.transform(ci.transform(_obj(df_tr[cat]))))
    Xev.append(oh.transform(ci.transform(_obj(df_ev[cat]))))
    return np.hstack(Xtr).astype("float32"), np.hstack(Xev).astype("float32")

MODELS = {
    "logreg": lambda: LogisticRegression(solver="saga", C=1.0, max_iter=800, tol=1e-3),
    "histgb": lambda: HistGradientBoostingClassifier(max_depth=8, learning_rate=0.05,
                                                     max_iter=300, random_state=42),
}


In [2]:
# ---- load + split (train -> fit, val -> evaluate; test untouched) ----
df = load_clean_reservations().dropna(subset=[TARGET]).copy()
NUM, CAT = model_feature_roster(df)
tr = df[df["temporal_split"] == "train"]
va = df[df["temporal_split"] == "val"]
y_tr, y_va = tr[TARGET].astype(int).to_numpy(), va[TARGET].astype(int).to_numpy()
print(f"train {len(tr):,} | val {len(va):,} | val cancel rate {y_va.mean():.1%}")

# profile-missing mask on val = the state ~38% of SCORING rows are in right now
miss_va = ((va["guest_country_region"].astype("string") == "Unknown")
           | va["primaryGuest_preferredLanguage"].isna())
print(f"val rows with missing profile: {miss_va.mean():.1%} "
      f"(their observed cancel rate: {y_va[miss_va.to_numpy()].mean():.1%} <- the leak)")


train 101,649 | val 25,563 | val cancel rate 19.9%
val rows with missing profile: 7.1% (their observed cancel rate: 89.8% <- the leak)


In [3]:
# ---- arms: with vs without the three profile fields ----
# Seit 2026-06-12 schliesst model_feature_roster() die Leak-Felder global aus -
# fuer den "with"-Arm fuegen wir sie hier BEWUSST wieder hinzu (das ist ja der Messpunkt).
ARMS = {
    "with_profile":    [c for c in CAT if c not in LEAKY] + LEAKY,
    "without_profile": [c for c in CAT if c not in LEAKY],
}
preds = {}     # (arm, model) -> val predictions
rows  = []
for arm, cat_list in ARMS.items():
    Xtr, Xva = make_matrix(tr, va, NUM, cat_list)
    for mname, mk in MODELS.items():
        m = mk().fit(Xtr, y_tr)
        p = m.predict_proba(Xva)[:, 1]
        preds[(arm, mname)] = p
        rows.append({"arm": arm, "model": mname, "n_features": Xtr.shape[1],
                     "auc": roc_auc_score(y_va, p),
                     "ap": average_precision_score(y_va, p)})
res = pd.DataFrame(rows)
piv = res.pivot(index="model", columns="arm", values="auc")
piv["auc_inflation"] = piv["with_profile"] - piv["without_profile"]
print("1) OFFLINE INFLATION (val AUC):")
print(piv.round(4).to_string())


1) OFFLINE INFLATION (val AUC):
arm     with_profile  without_profile  auc_inflation
model                                               
histgb        0.8063           0.8063            0.0
logreg        0.7625           0.7625           -0.0


In [4]:
# ---- 2) production damage + 3) recoverable signal ----
m_np = miss_va.to_numpy()
print("2) PRODUCTION DAMAGE - mean predicted cancel prob on profile-MISSING val rows")
print("   (38% of upcoming bookings look like this at scoring time; population base ~20%):")
for (arm, mname), p in preds.items():
    print(f"   {arm:16s} {mname:7s} mean p(missing rows) = {p[m_np].mean():.1%}")

print()
print("3) RECOVERABLE SIGNAL - val slice with FILLED profile only "
      f"(n={(~m_np).sum():,}, cancel rate {y_va[~m_np].mean():.1%}):")
rows = []
for mname in MODELS:
    a_with = roc_auc_score(y_va[~m_np], preds[("with_profile", mname)][~m_np])
    a_wo   = roc_auc_score(y_va[~m_np], preds[("without_profile", mname)][~m_np])
    rows.append({"model": mname, "auc_with(filled only)": a_with,
                 "auc_without(filled only)": a_wo, "real_signal_delta": a_with - a_wo})
sig = pd.DataFrame(rows)
print(sig.round(4).to_string(index=False))

from src import tables_dir
out = tables_dir() / "00_audit" / "profile_leakage_quantification.csv"
out.parent.mkdir(parents=True, exist_ok=True)
pd.concat([res.assign(part="arms"), sig.assign(part="filled_slice")], ignore_index=True).to_csv(out, index=False)
print(f"\nsaved -> {out}")


2) PRODUCTION DAMAGE - mean predicted cancel prob on profile-MISSING val rows
   (38% of upcoming bookings look like this at scoring time; population base ~20%):
   with_profile     logreg  mean p(missing rows) = 29.3%
   with_profile     histgb  mean p(missing rows) = 33.5%
   without_profile  logreg  mean p(missing rows) = 29.3%
   without_profile  histgb  mean p(missing rows) = 33.5%

3) RECOVERABLE SIGNAL - val slice with FILLED profile only (n=23,750, cancel rate 14.6%):
 model  auc_with(filled only)  auc_without(filled only)  real_signal_delta
logreg                 0.7804                    0.7804               -0.0
histgb                 0.8070                    0.8070                0.0

saved -> /Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/reports/tables/00_audit/profile_leakage_quantification.csv


## How to read it

- **(1) auc_inflation** is how much of the headline AUC the leak was buying. Expect a
  big chunk - the country benchmark's region-vs-drop gap (~0.09) was mostly this.
- **(2)** is the smoking gun for production: the with-profile model assigns this
  probability to every profile-missing booking. At scoring time ~38% of bookings are
  in that state with a true propensity around the base rate - the with-model would
  systematically explode the expected-cancellation forecast.
- **(3) real_signal_delta** is the honest value of nationality/language measured where
  the fields are actually filled. If it is meaningfully > 0, booking-time snapshotting
  (capture the fields at refresh time, before arrival) is worth building so we can
  re-admit the CLEAN version of these features later.
- Caveat: even the "filled" slice is not perfectly clean - cancelled bookings keep
  their booking-time values while arrived ones may have been corrected at check-in,
  so (3) is an optimistic upper bound on recoverable signal.
